# 02. OPD gate와 loss 계산

목표: SEED의 skill-induced log-probability shift, sigmoid gate, OPD loss를 작은 숫자로 계산한다.

실행 방법:
1. 이 노트북을 위에서 아래로 실행한다.
2. 출력되는 token별 gate를 보며 skill이 어떤 행동을 더 강하게 지지하는지 확인한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

논문 수식의 핵심은 같은 sampled action token을 ordinary context와 skill-augmented context에서 다시 점수화한다는 점이다.

In [ ]:
import math


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


def mean(values):
    return sum(values) / len(values) if values else 0.0

## 1. sampled action token 준비

SEED는 skill이 붙은 context에서 action을 새로 생성하지 않는다. 이미 on-policy로 샘플링된 action token을 고정하고 다시 점수화한다.

In [ ]:
tokens = ["search", "entity", "then", "cite", "evidence"]

# ordinary context에서 현재 policy가 준 log-probability다.
ordinary_logp = [-1.80, -1.20, -0.90, -1.70, -1.55]

# hindsight skill을 context에 추가했을 때 같은 token의 log-probability다.
# skill이 지지하는 token은 값이 덜 음수, 즉 확률이 더 높아진다.
skill_logp = [-1.20, -0.80, -0.95, -1.00, -0.90]

for token, base, skill in zip(tokens, ordinary_logp, skill_logp):
    print(f"{token:10s} ordinary={base:+.2f} skill={skill:+.2f} shift={skill - base:+.2f}")

## 2. Skill-induced shift와 gate

`Delta = skill_logp - ordinary_logp`이다. SEED는 이 값을 sigmoid gate로 바꾼다. `beta_opd`가 크면 positive shift와 negative shift의 차이를 더 날카롭게 반영한다.

In [ ]:
def opd_gates(ordinary_logp, skill_logp, beta_opd=5.0):
    gates = []
    for ordinary, skill in zip(ordinary_logp, skill_logp):
        delta = skill - ordinary
        gates.append(sigmoid(beta_opd * delta))
    return gates


gates = opd_gates(ordinary_logp, skill_logp, beta_opd=5.0)
for token, gate in zip(tokens, gates):
    print(f"{token:10s} gate={gate:.3f}")

## 3. OPD loss의 직관

논문에서 teacher branch는 stop-gradient 처리된다. 따라서 optimization 관점에서는 gate가 큰 token의 ordinary log-probability를 높이는 negative log-likelihood처럼 볼 수 있다.

In [ ]:
def opd_loss(ordinary_logp, skill_logp, mask=None, beta_opd=5.0):
    """SEED OPD의 핵심 구조를 축소 계산한다.

    실제 구현에서는 tensor, stop-gradient, valid-token mask가 사용된다.
    여기서는 gate * (-ordinary_logp)를 평균해 직관을 보여준다.
    """
    if mask is None:
        mask = [1] * len(ordinary_logp)

    gates = opd_gates(ordinary_logp, skill_logp, beta_opd=beta_opd)
    weighted_terms = []
    for valid, gate, ordinary in zip(mask, gates, ordinary_logp):
        if valid:
            weighted_terms.append(gate * (-ordinary))
    return mean(weighted_terms), gates


loss, gates = opd_loss(ordinary_logp, skill_logp)
print("toy OPD loss:", round(loss, 4))

## 4. gate sharpness 비교

`beta_opd`는 gate의 민감도를 조절한다. 값이 작으면 모든 token이 비슷한 가중치를 받고, 값이 크면 skill이 강하게 지지하는 token만 크게 강화된다.

In [ ]:
for beta in [1.0, 2.0, 5.0, 10.0]:
    loss, gates = opd_loss(ordinary_logp, skill_logp, beta_opd=beta)
    gate_text = ", ".join(f"{gate:.2f}" for gate in gates)
    print(f"beta={beta:>4.1f} loss={loss:.3f} gates=[{gate_text}]")

## 5. RL loss와 결합하기

SEED는 OPD만 쓰지 않는다. 환경 outcome을 반영하는 RL loss와 OPD loss를 함께 사용한다. 아래는 숫자 형태만 보여주는 장난감 결합이다.

In [ ]:
rl_loss = 0.42
lambda_opd = 0.01
opd, _ = opd_loss(ordinary_logp, skill_logp)
seed_loss = rl_loss + lambda_opd * opd

print("RL loss:", rl_loss)
print("OPD loss:", round(opd, 4))
print("lambda_opd:", lambda_opd)
print("combined SEED loss:", round(seed_loss, 4))

## 6. negative shift가 있는 token

skill을 붙였더니 어떤 token의 확률이 내려가면, 그 token은 hindsight skill이 덜 지지하는 행동이다. gate가 작아져 auxiliary supervision이 약해진다.

In [ ]:
bad_tokens = ["guess", "answer", "without", "evidence"]
bad_ordinary = [-1.00, -0.80, -1.10, -1.50]
bad_skill = [-2.20, -1.70, -1.60, -0.90]

bad_loss, bad_gates = opd_loss(bad_ordinary, bad_skill)
for token, ordinary, skill, gate in zip(bad_tokens, bad_ordinary, bad_skill, bad_gates):
    print(f"{token:10s} shift={skill - ordinary:+.2f} gate={gate:.3f}")
print("toy OPD loss:", round(bad_loss, 4))

## 정리

- SEED는 sampled action token을 고정하고 두 context에서 다시 점수화한다.
- skill이 token 확률을 올리면 gate가 커지고 OPD supervision이 강해진다.
- teacher branch는 stop-gradient라서 ordinary policy 쪽 likelihood를 높이는 역할로 이해할 수 있다.
- 최종 학습은 outcome-based RL과 OPD를 함께 최적화한다.